# Web-Gold-40K — recovery v2.2 bbox correction

This isolated notebook preserves recovery-v2.1 and changes only bbox geometry/parameterization justified by its full-screen collapse. Run `audit`, `smoke`, `bbox_overfit`, `diagnostic`, then `mini`, restarting the kernel between stages. Do not skip a gate.

In [1]:
# 1. Pull the latest modular code and record the environment.
from pathlib import Path
import importlib.metadata as metadata
import json, os, subprocess, sys
REPOSITORY = 'https://github.com/Kiyas-Mahmud/webagent.git'
REPO_ROOT = Path('/kaggle/working/webagent')
SOURCE_ROOT = REPO_ROOT / 'src'
if (REPO_ROOT / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', 'Code'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'Code', '--single-branch', REPOSITORY, str(REPO_ROOT)], check=True)
requirements = ['transformers>=4.49,<5', 'peft>=0.14,<1', 'bitsandbytes>=0.45,<1', 'accelerate>=1,<2', 'scikit-learn>=1.4,<2']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *requirements], check=True)
for module_name in list(sys.modules):
    if module_name == 'web_agent' or module_name.startswith('web_agent.'):
        del sys.modules[module_name]
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(REPO_ROOT)
environment = {name: metadata.version(name) for name in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate', 'scikit-learn']}
environment['python'] = sys.version.split()[0]
environment['git_commit'] = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
Path('/kaggle/working/gold_recovery_v2_2_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))

Cloning into '/kaggle/working/webagent'...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
{
  "torch": "2.10.0+cu128",
  "transformers": "4.57.6",
  "peft": "0.19.1",
  "bitsandbytes": "0.49.2",
  "accelerate": "1.14.0",
  "scikit-learn": "1.9.0",
  "python": "3.12.13",
  "git_commit": "9e21edb8562f40ad9932aa08161a05a02456f3a5"
}


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.0 which is incompatible.


In [2]:
# 2. Locate the attached dataset without downloading or extracting it.
ATTACHED_ROOT = Path('/kaggle/input/datasets/kiyasmahmud/web-gold-40k')
SPLIT_FILES = ('split_train.json', 'split_val.json', 'split_test.json')
def find_split_root(root: Path) -> Path:
    if all((root / name).is_file() for name in SPLIT_FILES):
        return root
    candidates = []
    for current, _, files in os.walk(root, followlinks=True):
        if set(SPLIT_FILES).issubset(files):
            candidates.append(Path(current))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one structured split folder; found {candidates}')
    return candidates[0]
DATA_ROOT = find_split_root(ATTACHED_ROOT).resolve()
print('DATA_ROOT =', DATA_ROOT)

DATA_ROOT = /kaggle/input/datasets/kiyasmahmud/web-gold-40k/final_data_set_40k


In [3]:
# 3. Select exactly one gate. Restart the kernel before changing stages.
import torch
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
STAGE = 'audit'  # 'audit', 'smoke', 'bbox_overfit', 'diagnostic', or 'mini'
SEED = 42
TRAIN_ROWS, VAL_ROWS = 5_000, 500
EPOCHS = 1 if STAGE == 'diagnostic' else 5
assert STAGE in {'audit', 'smoke', 'bbox_overfit', 'diagnostic', 'mini'}
if STAGE != 'audit':
    assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator first.'
set_seed(SEED)
cfg = load_config('configs/backbones/qwen2vl_2b_gold_v2_2.yaml')
cfg['data']['root'] = str(DATA_ROOT)
cfg['data']['num_workers'] = 0 if STAGE in {'audit', 'smoke', 'bbox_overfit'} else 4
assert cfg['data']['strict_bbox_geometry']
assert cfg['model']['bbox_parameterization'] == 'cxcywh'
assert cfg['loss']['bbox_loss'] == 'detr_l1_giou'
assert cfg['loss']['bbox_l1_ratio'] == 5.0 and cfg['loss']['bbox_giou_ratio'] == 2.0
print('stage:', STAGE, '| epochs:', EPOCHS)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

stage: audit | epochs: 5
GPU: Tesla T4


In [4]:
# 4. Always audit complete train/validation geometry before a training gate.
from web_agent.train.gold_stages import run_gold_bbox_audit
bbox_audit = run_gold_bbox_audit(cfg)
audit_path = Path('/kaggle/working/gold_recovery_v2_2_bbox_audit.json')
audit_path.write_text(json.dumps(bbox_audit, indent=2), encoding='utf-8')
print(json.dumps(bbox_audit, indent=2))
print('BBox audit:', audit_path)
assert bbox_audit['status'] == 'PASS', 'Fix only the reported invalid bbox annotations, then rerun audit.'

{
  "status": "FAIL",
  "test_rows_read": 0,
  "train": {
    "status": "FAIL",
    "records": 23499,
    "bbox_rows": 8761,
    "valid_bbox_rows": 6719,
    "invalid_bbox_rows": 2042,
    "invalid_reason_counts": {
      "bottom_boundary_overflow": 1835,
      "negative_origin": 95,
      "non_positive_size": 6,
      "right_boundary_overflow": 115,
      "x_origin_outside": 72,
      "y_origin_outside": 1755
    },
    "invalid_examples": [
      {
        "record_id": "gold_v16_40k_000011__step_0000",
        "state_before": "images/gold_v16_40k_000011/before_0001.png",
        "image_size": [
          1280,
          720
        ],
        "bbox": {
          "x": 527.609375,
          "y": 750.0,
          "width": 53.625,
          "height": 12.0
        },
        "reasons": [
          "bottom_boundary_overflow",
          "y_origin_outside"
        ]
      },
      {
        "record_id": "gold_v16_40k_000013__step_0000",
        "state_before": "images/gold_v16_40k_000013/bef

AssertionError: Fix only the reported invalid bbox annotations, then rerun audit.

In [ ]:
# 5. Run the selected gate. No stage opens split_test.json.
from web_agent.train.gold_stages import build_processor, run_gold_bbox_overfit, run_gold_mini, run_gold_smoke
from web_agent.utils.results import save_mini_diagnostics_json, save_mini_result_csv
processor = None
if STAGE == 'audit':
    stage_report = bbox_audit
else:
    processor = build_processor(cfg)
    if STAGE == 'smoke':
        stage_report = run_gold_smoke(cfg, processor=processor, rows=16, seed=SEED)
    elif STAGE == 'bbox_overfit':
        stage_report = run_gold_bbox_overfit(cfg, processor=processor, seed=SEED)
    else:
        stage_report = run_gold_mini(cfg, processor=processor, train_rows=TRAIN_ROWS, val_rows=VAL_ROWS, epochs=EPOCHS, seed=SEED)
report_path = Path(f'/kaggle/working/gold_recovery_v2_2_{STAGE}_report.json')
report_path.write_text(json.dumps(stage_report, indent=2), encoding='utf-8')
result_csv_path = diagnostics_path = None
if STAGE in {'diagnostic', 'mini'}:
    result_csv_path = save_mini_result_csv(stage_report, f'/kaggle/working/gold_recovery_v2_2_{STAGE}_result.csv')
    diagnostics_path = save_mini_diagnostics_json(stage_report, f'/kaggle/working/gold_recovery_v2_2_{STAGE}_diagnostics.json')
print(json.dumps(stage_report, indent=2))
print('Report:', report_path, '| CSV:', result_csv_path, '| diagnostics:', diagnostics_path)

In [ ]:
# 6. Enforce the next permitted action.
assert stage_report['status'] == 'PASS'
assert stage_report['test_rows_read'] == 0
if STAGE == 'audit':
    print('AUDIT PASSED. Restart, set STAGE to smoke, then Run All.')
elif STAGE == 'smoke':
    for name in ('bbox', 'needs_recovery', 'strategy', 'recovery_outcome', 'grounding_adapter'):
        assert stage_report['probe_gradient_norms'][name] > 0, f'No gradient: {name}'
        assert stage_report['probe_update_norms'][name] > 0, f'No update: {name}'
    assert stage_report['bbox_supervised_rows'] > 0
    assert min(stage_report['spatial_tokens_per_row']) > 0
    print('SMOKE PASSED. Restart, set STAGE to bbox_overfit, then Run All.')
elif STAGE == 'bbox_overfit':
    assert all(stage_report['checks'].values())
    print('BBOX OVERFIT PASSED:', stage_report['initial']['bbox_mean_iou'], '->', stage_report['final']['bbox_mean_iou'])
    print('Restart, set STAGE to diagnostic, then Run All.')
else:
    assert len(stage_report['history']) == EPOCHS
    assert stage_report['checkpoint_roundtrip']
    assert result_csv_path.is_file() and diagnostics_path.is_file()
    last = stage_report['history'][-1]
    print('bbox:', last['bbox_mean_iou'], last['bbox_recall_iou50'])
    print('bbox L1/GIoU:', last['train_raw_bbox_l1_loss'], last['train_raw_bbox_giou_loss'])
    print('needs recovery:', last['needs_recovery_macro_f1'], 'strategy:', last['strategy_attempted_macro_f1'])
    if STAGE == 'diagnostic':
        required = ('needs_recovery_macro_f1_beats_majority_by_0_03', 'attempted_strategy_macro_f1_beats_majority_by_0_03', 'transition_recovery_outcome_mcc_improves_v14_by_0_01', 'bbox_mean_iou_at_least_0_05', 'bbox_recall_iou50_at_least_0_01', 'outcome_ece_not_up_more_than_0_03')
        checks = stage_report['quality_gates']['checks']
        assert all(checks[name] for name in required), {name: checks[name] for name in required}
        print('DIAGNOSTIC FUNCTIONAL GATES PASSED. Restart, set STAGE to mini, then Run All.')
    else:
        assert stage_report['loss_decreased'] is True
        assert len(stage_report['epoch_checkpoints']) == EPOCHS
        assert stage_report['quality_gates']['status'] == 'PASS'
        print('FIVE-EPOCH QUALITY GATES PASSED. Preserve every artifact; full training remains separately gated.')

## Decision rule

An audit failure means the reported annotations must be reviewed; the notebook never repairs thesis data silently. A micro-overfit failure blocks the expensive diagnostic because the bbox architecture has not proved trainability. The one-epoch diagnostic must pass localization, recovery, and calibration functionality before the five-epoch mini. Only the five-epoch mini applies every outcome/action quality gate.